# F1 Pit Stop Prediction - Exploratory Data Analysis
**Kaggle Playground Series S6E5**

Goal: understand the data before we touch any models. We want to know distributions, class imbalance, correlations, and spot any leakage risks early.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

pd.set_option('display.max_columns', None)
sns.set_theme(style='darkgrid')

train = pd.read_csv('../data/train.csv')
test  = pd.read_csv('../data/test.csv')

print('Train shape:', train.shape)
print('Test shape: ', test.shape)

## 1. Basic look at the data

In [ ]:
train.head(10)

In [ ]:
train.info()

In [ ]:
train.describe()

## 2. Missing values

In [ ]:
missing = train.isnull().sum()
print('Missing values in train:')
print(missing[missing > 0] if missing.any() else 'None')

## 3. Target distribution

In [ ]:
target_counts = train['PitNextLap'].value_counts()
print(target_counts)
print(f'\nClass ratio: {target_counts[0]/target_counts[1]:.2f} : 1 (no pit : pit)')

fig, ax = plt.subplots(figsize=(6, 4))
target_counts.plot(kind='bar', ax=ax, color=['steelblue', 'tomato'])
ax.set_title('Target distribution: PitNextLap')
ax.set_xticklabels(['No Pit (0)', 'Pit (1)'], rotation=0)
ax.set_ylabel('Count')
plt.tight_layout()
plt.show()

## 4. Categorical features

In [ ]:
cat_cols = ['Driver', 'Compound', 'Race']
for col in cat_cols:
    print(f'{col}: {train[col].nunique()} unique values')
    print(train[col].value_counts().head(10))
    print()

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
for ax, col in zip(axes, cat_cols):
    pit_rate = train.groupby(col)['PitNextLap'].mean().sort_values(ascending=False)
    pit_rate.head(20).plot(kind='bar', ax=ax)
    ax.set_title(f'Pit rate by {col} (top 20)')
    ax.set_ylabel('Pit probability')
    ax.tick_params(axis='x', rotation=45)
plt.tight_layout()
plt.show()

## 5. Tyre life by compound

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for pit_val, ax, label in zip([0, 1], axes, ['No Pit Next Lap', 'Pit Next Lap']):
    subset = train[train['PitNextLap'] == pit_val]
    for compound in train['Compound'].unique():
        data = subset[subset['Compound'] == compound]['TyreLife']
        ax.hist(data, bins=30, alpha=0.5, label=compound)
    ax.set_title(f'TyreLife distribution ({label})')
    ax.set_xlabel('TyreLife (laps)')
    ax.legend()

plt.tight_layout()
plt.show()

In [ ]:
print('Median TyreLife at pit by compound:')
print(train[train['PitNextLap'] == 1].groupby('Compound')['TyreLife'].describe())

## 6. Numerical feature correlations with target

In [ ]:
num_cols = ['LapNumber', 'Stint', 'TyreLife', 'Position', 'LapTime (s)',
            'LapTime_Delta', 'Cumulative_Degradation', 'RaceProgress', 'Position_Change']

corr = train[num_cols + ['PitNextLap']].corr()['PitNextLap'].drop('PitNextLap').sort_values()
print('Correlation with PitNextLap:')
print(corr)

In [ ]:
fig, ax = plt.subplots(figsize=(8, 6))
corr.plot(kind='barh', ax=ax, color=['tomato' if x < 0 else 'steelblue' for x in corr])
ax.set_title('Feature correlation with PitNextLap')
ax.axvline(0, color='black', linewidth=0.8)
plt.tight_layout()
plt.show()

## 7. Pit rate across race progress

In [ ]:
train['RaceProgress_bin'] = pd.cut(train['RaceProgress'], bins=20)
pit_by_progress = train.groupby('RaceProgress_bin', observed=True)['PitNextLap'].mean()

fig, ax = plt.subplots(figsize=(12, 4))
pit_by_progress.plot(kind='bar', ax=ax, color='steelblue')
ax.set_title('Pit probability across race progress')
ax.set_ylabel('Pit probability')
ax.set_xlabel('Race progress (binned)')
ax.tick_params(axis='x', rotation=45)
plt.tight_layout()
plt.show()

train.drop(columns=['RaceProgress_bin'], inplace=True)

## 8. Year distribution

In [ ]:
print('Rows per year:')
print(train.groupby('Year').size())
print('\nPit rate per year:')
print(train.groupby('Year')['PitNextLap'].mean())

## 9. Leakage check

PitStop column tells us if a pit happened on the current lap. If PitStop=1 then PitNextLap should almost never be 1 (you do not pit two laps in a row). Let's verify this.

In [ ]:
print('PitStop vs PitNextLap crosstab:')
print(pd.crosstab(train['PitStop'], train['PitNextLap'], normalize='index'))

## 10. Train vs Test feature distributions

Quick sanity check that test looks like train.

In [ ]:
fig, axes = plt.subplots(3, 3, figsize=(16, 12))
axes = axes.flatten()

for i, col in enumerate(num_cols):
    axes[i].hist(train[col], bins=40, alpha=0.5, label='Train', density=True, color='steelblue')
    axes[i].hist(test[col],  bins=40, alpha=0.5, label='Test',  density=True, color='tomato')
    axes[i].set_title(col)
    axes[i].legend(fontsize=8)

plt.suptitle('Train vs Test distributions', y=1.01, fontsize=14)
plt.tight_layout()
plt.show()

## EDA Summary

Fill this in after running the cells above:

- Target imbalance: 
- Most important raw features (by correlation): 
- Tyre life at pit window (by compound): 
- Any leakage found: 
- Train vs test distribution issues: 